# Step D1 — Dataset prep (build manifests + pos_weight.json)

In [14]:

import pandas as pd, numpy as np, json
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit

ROOT = Path("/kaggle/input")
csv_candidates = list(ROOT.rglob("Data_Entry_2017*.csv"))
assert csv_candidates, "❌ Data_Entry_2017.csv not found under /kaggle/input"
META_CSV = csv_candidates[0]
print("Using metadata:", META_CSV)

# find nested image folders
nested_dirs = [d/"images" for d in META_CSV.parent.iterdir()
               if d.is_dir() and d.name.startswith("images_") and (d/"images").is_dir()]
assert nested_dirs, "❌ No nested image folders found"

# map filenames -> paths
index = {}
for d in nested_dirs:
    for p in d.glob("*.png"):
        index[p.name] = str(p)

CLASSES = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass','Nodule',
    'Pneumonia','Pneumothorax','Consolidation','Edema','Emphysema',
    'Fibrosis','Pleural_Thickening','Hernia'
]
ALIASES = {"Pleural Thickening":"Pleural_Thickening","No Finding":"No Finding"}

def encode(lbls):
    if lbls.strip()=="No Finding": return np.zeros(len(CLASSES),dtype=int)
    parts=[ALIASES.get(s.strip(),s.strip()) for s in lbls.split('|')]
    vec=np.zeros(len(CLASSES),dtype=int)
    for i,c in enumerate(CLASSES):
        if c in parts: vec[i]=1
    return vec

df = pd.read_csv(META_CSV)
df["image_path"] = df["Image Index"].map(index.get)
df = df[df["image_path"].notna()].copy()
lab = np.stack(df["Finding Labels"].apply(encode).to_numpy())
for i,c in enumerate(CLASSES): df[c]=lab[:,i]

# patient-wise 70/15/15
groups = df["Patient ID"].values
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_val_idx, test_idx = next(gss1.split(df, groups=groups))
df_tv, df_te = df.iloc[train_val_idx].copy(), df.iloc[test_idx].copy()
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
train_idx, val_idx = next(gss2.split(df_tv, groups=df_tv["Patient ID"].values))
df_tr, df_va = df_tv.iloc[train_idx].copy(), df_tv.iloc[val_idx].copy()

df_tr[["image_path","Patient ID"]+CLASSES].to_csv("/kaggle/working/train.csv",index=False)
df_va[["image_path","Patient ID"]+CLASSES].to_csv("/kaggle/working/val.csv",index=False)
df_te[["image_path","Patient ID"]+CLASSES].to_csv("/kaggle/working/test.csv",index=False)

# pos_weight
pos = df_tr[CLASSES].sum(0).values
neg = len(df_tr)-pos
pw = (neg/np.clip(pos,1,None)).astype(float)
with open("/kaggle/working/pos_weight.json","w") as f:
    json.dump({c:float(w) for c,w in zip(CLASSES,pw)}, f, indent=2)

print("Splits built:")
print(" Train:",len(df_tr)," Val:",len(df_va)," Test:",len(df_te))
print("pos_weight.json saved")


Using metadata: /kaggle/input/data/Data_Entry_2017.csv
Splits built:
 Train: 78298  Val: 17098  Test: 16724
pos_weight.json saved


# Step D2-fix — remove 1000-class head from the checkpoint, then load backbone

In [16]:

import torch, timm

CLASSES = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass','Nodule',
    'Pneumonia','Pneumothorax','Consolidation','Edema','Emphysema',
    'Fibrosis','Pleural_Thickening','Hernia'
]

WEIGHTS_PATH = "/kaggle/input/test-2-ds/vit_tiny_patch16_224_in1k.pth"

# 1) Build model with 14-class head
model = timm.create_model("vit_tiny_patch16_224", pretrained=False, num_classes=len(CLASSES))

# 2) Load checkpoint and strip classifier head params
state = torch.load(WEIGHTS_PATH, map_location="cpu")
clean = {}
for k, v in state.items():
    nk = k
    if nk.startswith("model."):  nk = nk[6:]
    if nk.startswith("module."): nk = nk[7:]
    # drop any classifier head weights/bias from the 1000-class model
    if nk.startswith("head."):
        continue
    clean[nk] = v

# 3) Load backbone only (strict=False lets our new 14-class head remain randomly init)
msg = model.load_state_dict(clean, strict=False)
print("Loaded (backbone only):", msg)

# 4) Sanity forward
with torch.no_grad():
    y = model(torch.randn(2, 3, 224, 224))
print("Output shape:", tuple(y.shape))  # should be (2, 14)


Loaded (backbone only): _IncompatibleKeys(missing_keys=['head.weight', 'head.bias'], unexpected_keys=[])
Output shape: (2, 14)


# Step D3 — Transforms + DataLoaders + one-batch sanity (no training yet)

In [17]:

import os, json, cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A

# Silence albumentations version ping
os.environ["ALBUMENTATIONS_DISABLE_VERSION_CHECK"] = "1"

# Paths from earlier
TRAIN_CSV = "/kaggle/working/train.csv"
VAL_CSV   = "/kaggle/working/val.csv"
TEST_CSV  = "/kaggle/working/test.csv"

CLASSES = [
    'Atelectasis','Cardiomegaly','Effusion','Infiltration','Mass','Nodule',
    'Pneumonia','Pneumothorax','Consolidation','Edema','Emphysema',
    'Fibrosis','Pleural_Thickening','Hernia'
]
IMG_SIZE, BATCH_SIZE = 224, 16
device = torch.device("cpu")

# Transforms (conservative, ViT-friendly)
train_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_AREA),
    A.HorizontalFlip(p=0.5),
    A.Affine(rotate=(-8,8), translate_percent={"x":0.02,"y":0.02}, scale=(0.9,1.1), p=0.3),
    A.RandomBrightnessContrast(p=0.15),
    A.Normalize(mean=(0.485,), std=(0.229,)),
])
val_tfms = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE, interpolation=cv2.INTER_AREA),
    A.Normalize(mean=(0.485,), std=(0.229,)),
])

class CXRDataset(Dataset):
    def __init__(self, csv_path, is_train=True):
        self.df = pd.read_csv(csv_path)
        self.tfms = train_tfms if is_train else val_tfms
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        img = cv2.imread(r["image_path"], cv2.IMREAD_GRAYSCALE)
        if img is None: raise FileNotFoundError(r["image_path"])
        img = img[..., None]
        img = self.tfms(image=img)["image"]
        x = torch.tensor(np.transpose(np.repeat(img, 3, axis=2), (2,0,1)), dtype=torch.float32)
        y = torch.tensor(r[CLASSES].values.astype(np.float32))
        return x, y

# Build loaders
train_loader = DataLoader(CXRDataset(TRAIN_CSV, True), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(CXRDataset(VAL_CSV,   False), batch_size=BATCH_SIZE*2, shuffle=False, num_workers=0)

# Load pos_weight
with open("/kaggle/working/pos_weight.json","r") as f:
    pw = json.load(f)
pos_weight = torch.tensor([pw[c] for c in CLASSES], dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# One-batch sanity: shapes + forward + loss
xb, yb = next(iter(train_loader))
with torch.no_grad():
    logits = model(xb.to(device))
    loss = criterion(logits, yb.to(device))
print("Batch shapes:", tuple(xb.shape), tuple(yb.shape))
print("Logits shape:", tuple(logits.shape))
print("Sanity BCE loss:", float(loss))


Batch shapes: (16, 3, 224, 224) (16, 14)
Logits shape: (16, 14)
Sanity BCE loss: 1.153515100479126


# Step D4 — Training (head-only → finetune), early stop on val micro-AUC

In [18]:

import time, numpy as np, torch, torch.nn as nn
from sklearn.metrics import roc_auc_score, f1_score

device = torch.device("cpu")
CKPT_PATH = "/kaggle/working/vittiny_best.pt"

# --- knobs you can tweak later ---
HEAD_EPOCHS   = 2
FT_EPOCHS     = 6
PATIENCE      = 2
MAX_TRAIN_STEPS = 300    # set None for full epoch; increase later for better results
LR_HEAD, WD_HEAD = 1e-3, 1e-4
LR_FT,   WD_FT   = 1e-4, 5e-2

# reuse 'model', 'criterion', 'train_loader', 'val_loader' from earlier cell

def evaluate(model, loader, thr=0.5):
    model.eval()
    P, Y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device)).cpu().numpy()
            probs  = 1/(1+np.exp(-logits))
            P.append(probs); Y.append(yb.numpy())
    P, Y = np.vstack(P), np.vstack(Y)
    micro_auc = roc_auc_score(Y, P, average='micro')
    preds = (P >= thr).astype(int)
    micro_f1 = f1_score(Y, preds, average='micro', zero_division=0)
    return float(micro_auc), float(micro_f1)

def train_one_epoch(loader, optimizer, step_cap=None):
    model.train()
    total, n, steps = 0.0, 0, 0
    for xb, yb in loader:
        logits = model(xb.to(device))
        loss   = criterion(logits, yb.to(device))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total += loss.item()*xb.size(0)
        n     += xb.size(0)
        steps += 1
        if step_cap and steps >= step_cap:
            break
    return total/max(1,n)

def run_phase(name, epochs, optimizer):
    best_auc, bad = -1.0, 0
    for ep in range(1, epochs+1):
        t0 = time.time()
        tr_loss = train_one_epoch(train_loader, optimizer, step_cap=MAX_TRAIN_STEPS)
        val_auc, val_f1 = evaluate(model, val_loader, thr=0.5)
        mins = (time.time()-t0)/60
        print(f"[{name}] Ep{ep:02d} | loss={tr_loss:.4f} | val micro-AUC={val_auc:.4f} | micro-F1={val_f1:.4f} | {mins:.1f}m")
        if val_auc > best_auc:
            best_auc, bad = val_auc, 0
            torch.save({"state_dict": model.state_dict()}, CKPT_PATH)
            print("  ✅ New best — saved to", CKPT_PATH)
        else:
            bad += 1
            if bad >= PATIENCE:
                print(f"  ⏹ Early stop (no AUC improve {PATIENCE} epochs). Best={best_auc:.4f}")
                break
    return best_auc

# ---- Phase 1: freeze backbone, train head only ----
for n,p in model.named_parameters():
    p.requires_grad = n.startswith("head.")
opt_head = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                             lr=LR_HEAD, weight_decay=WD_HEAD)
print("=== Phase 1: Head-only ===")
best1 = run_phase("head", HEAD_EPOCHS, opt_head)

# ---- Phase 2: unfreeze all, fine-tune ----
for p in model.parameters(): p.requires_grad = True
opt_ft = torch.optim.AdamW(model.parameters(), lr=LR_FT, weight_decay=WD_FT)
print("=== Phase 2: Fine-tune ===")
best2 = run_phase("finetune", FT_EPOCHS, opt_ft)

print("Training done. Best val micro-AUC observed:", round(max(best1, best2), 4))


=== Phase 1: Head-only ===
[head] Ep01 | loss=1.4416 | val micro-AUC=0.6574 | micro-F1=0.1413 | 19.8m
  ✅ New best — saved to /kaggle/working/vittiny_best.pt
[head] Ep02 | loss=1.3138 | val micro-AUC=0.6974 | micro-F1=0.1559 | 16.7m
  ✅ New best — saved to /kaggle/working/vittiny_best.pt
=== Phase 2: Fine-tune ===
[finetune] Ep01 | loss=1.3582 | val micro-AUC=0.6173 | micro-F1=0.1240 | 22.0m
  ✅ New best — saved to /kaggle/working/vittiny_best.pt
[finetune] Ep02 | loss=1.2803 | val micro-AUC=0.6307 | micro-F1=0.1275 | 21.5m
  ✅ New best — saved to /kaggle/working/vittiny_best.pt
[finetune] Ep03 | loss=1.2247 | val micro-AUC=0.6774 | micro-F1=0.1371 | 21.9m
  ✅ New best — saved to /kaggle/working/vittiny_best.pt


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


[finetune] Ep04 | loss=1.2322 | val micro-AUC=0.7165 | micro-F1=0.1491 | 22.3m
  ✅ New best — saved to /kaggle/working/vittiny_best.pt
[finetune] Ep05 | loss=1.2264 | val micro-AUC=0.7319 | micro-F1=0.1775 | 21.1m
  ✅ New best — saved to /kaggle/working/vittiny_best.pt
[finetune] Ep06 | loss=1.2255 | val micro-AUC=0.6910 | micro-F1=0.1478 | 20.9m
Training done. Best val micro-AUC observed: 0.7319


# Step D5 — threshold tuning (val) + final test eval (uses best ckpt)

In [19]:

import json, numpy as np, torch
from sklearn.metrics import roc_auc_score, f1_score

CKPT_PATH = "/kaggle/working/vittiny_best.pt"
device = torch.device("cpu")

# load best weights into the existing `model`
sd = torch.load(CKPT_PATH, map_location="cpu")["state_dict"]
model.load_state_dict(sd, strict=False)
model.eval()

def predict(loader, tta=False):
    Ps, Ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            if tta:
                logits1 = model(xb.to(device)).cpu().numpy()
                logits2 = model(torch.flip(xb, dims=[3]).to(device)).cpu().numpy()
                probs = (1/(1+np.exp(-logits1)) + 1/(1+np.exp(-logits2))) / 2
            else:
                logits = model(xb.to(device)).cpu().numpy()
                probs = 1/(1+np.exp(-logits))
            Ps.append(probs); Ys.append(yb.numpy())
    return np.vstack(Ps), np.vstack(Ys)

def metrics(P, Y, thr):
    micro_auc = roc_auc_score(Y, P, average='micro')
    # macro AUC across classes with positives
    aucs=[]
    for j in range(P.shape[1]):
        if Y[:,j].sum()>0:
            try: aucs.append(roc_auc_score(Y[:,j], P[:,j]))
            except: pass
    macro_auc = float(np.mean(aucs)) if aucs else float("nan")
    preds = (P>=thr).astype(int) if isinstance(thr,float) else (P>=np.asarray(thr)[None,:]).astype(int)
    micro_f1 = f1_score(Y, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(Y, preds, average='macro', zero_division=0)
    return dict(microAUC=float(micro_auc), macroAUC=float(macro_auc),
                microF1=float(micro_f1), macroF1=float(macro_f1))

# 1) tune thresholds on VAL
P_val, Y_val = predict(val_loader, tta=False)
grid = np.linspace(0.05, 0.95, 19)
thr_vec = np.zeros(P_val.shape[1], dtype=np.float32)
for j in range(P_val.shape[1]):
    if Y_val[:,j].sum()==0: thr_vec[j]=0.5; continue
    f1s = [f1_score(Y_val[:,j], (P_val[:,j]>=t).astype(int), zero_division=0) for t in grid]
    thr_vec[j] = float(grid[int(np.argmax(f1s))])

val_base  = metrics(P_val, Y_val, thr=0.5)
val_tuned = metrics(P_val, Y_val, thr=thr_vec)
print(f"[VAL] base  microAUC={val_base['microAUC']:.4f} macroAUC={val_base['macroAUC']:.4f} microF1={val_base['microF1']:.4f} macroF1={val_base['macroF1']:.4f}")
print(f"[VAL] tuned microAUC={val_tuned['microAUC']:.4f} macroAUC={val_tuned['macroAUC']:.4f} microF1={val_tuned['microF1']:.4f} macroF1={val_tuned['macroF1']:.4f}")

# save thresholds
with open("/kaggle/working/thresholds.json","w") as f:
    json.dump({c: float(t) for c,t in zip(CLASSES, thr_vec)}, f, indent=2)

# 2) TEST with tuned thresholds
P_test, Y_test = predict(test_loader, tta=False)
test_tuned = metrics(P_test, Y_test, thr=thr_vec)
print(f"[TEST] tuned microAUC={test_tuned['microAUC']:.4f} macroAUC={test_tuned['macroAUC']:.4f} microF1={test_tuned['microF1']:.4f} macroF1={test_tuned['macroF1']:.4f}")

# optional TTA
P_test_tta, _ = predict(test_loader, tta=True)
test_tuned_tta = metrics(P_test_tta, Y_test, thr=thr_vec)
print(f"[TEST+TTA] tuned microAUC={test_tuned_tta['microAUC']:.4f} macroAUC={test_tuned_tta['macroAUC']:.4f} microF1={test_tuned_tta['microF1']:.4f} macroF1={test_tuned_tta['macroF1']:.4f}")

# dump metrics
with open("/kaggle/working/metrics_final.json","w") as f:
    json.dump({
        "val_base": val_base, "val_tuned": val_tuned,
        "test_tuned": test_tuned, "test_tuned_tta": test_tuned_tta,
        "thresholds": {c: float(t) for c,t in zip(CLASSES, thr_vec)}
    }, f, indent=2)
print("Saved metrics_final.json & thresholds.json in /kaggle/working")

[VAL] base  microAUC=0.7319 macroAUC=0.7248 microF1=0.1775 macroF1=0.1494
[VAL] tuned microAUC=0.7319 macroAUC=0.7248 microF1=0.2395 macroF1=0.1781
[TEST] tuned microAUC=0.7274 macroAUC=0.7192 microF1=0.2253 macroF1=0.1672
[TEST+TTA] tuned microAUC=0.7299 macroAUC=0.7223 microF1=0.2309 macroF1=0.1660
Saved metrics_final.json & thresholds.json in /kaggle/working


# Low-LR polish: resume from best, 2-3 short epochs at LR=5e-5

In [20]:

import torch, time, numpy as np
from sklearn.metrics import roc_auc_score, f1_score

CKPT_PATH = "/kaggle/working/vittiny_best.pt"
device = torch.device("cpu")

# reload best weights
sd = torch.load(CKPT_PATH, map_location="cpu")["state_dict"]
model.load_state_dict(sd, strict=False)
for p in model.parameters(): p.requires_grad = True

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=5e-2)
PATIENCE = 2
MAX_TRAIN_STEPS = 300  # raise/None later if you want full epochs

def evaluate(loader):
    model.eval()
    P, Y = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits = model(xb.to(device)).cpu().numpy()
            probs  = 1/(1+np.exp(-logits))
            P.append(probs); Y.append(yb.numpy())
    P, Y = np.vstack(P), np.vstack(Y)
    micro_auc = roc_auc_score(Y, P, average='micro')
    preds = (P>=0.5).astype(int)
    micro_f1 = f1_score(Y, preds, average='micro', zero_division=0)
    return float(micro_auc), float(micro_f1)

def train_one_epoch():
    model.train()
    tot, n, steps = 0.0, 0, 0
    for xb, yb in train_loader:
        logits = model(xb.to(device))
        loss = criterion(logits, yb.to(device))
        optimizer.zero_grad(set_to_none=True)
        loss.backward(); optimizer.step()
        tot += loss.item()*xb.size(0); n += xb.size(0)
        steps += 1
        if MAX_TRAIN_STEPS and steps >= MAX_TRAIN_STEPS: break
    return tot/max(1,n)

best_auc, bad = -1.0, 0
for ep in range(1, 4):  # 3 short epochs
    t0 = time.time()
    tr_loss = train_one_epoch()
    val_auc, val_f1 = evaluate(val_loader)
    print(f"[polish] Ep{ep:02d} | loss={tr_loss:.4f} | val micro-AUC={val_auc:.4f} | micro-F1={val_f1:.4f} | {(time.time()-t0)/60:.1f}m")
    if val_auc > best_auc:
        best_auc, bad = val_auc, 0
        torch.save({"state_dict": model.state_dict()}, CKPT_PATH)
        print("  ✅ New best saved:", CKPT_PATH)
    else:
        bad += 1
        if bad >= PATIENCE:
            print(f"  ⏹ Early stop (no AUC improve {PATIENCE} epochs). Best={best_auc:.4f}")
            break
print("Polish pass done.")


libpng warning: iCCP: profile 'ICC Profile': 'GRAY': Gray color space not permitted on RGB PNG


[polish] Ep01 | loss=1.1781 | val micro-AUC=0.6973 | micro-F1=0.1552 | 22.9m
  ✅ New best saved: /kaggle/working/vittiny_best.pt
[polish] Ep02 | loss=1.1553 | val micro-AUC=0.7799 | micro-F1=0.2002 | 22.1m
  ✅ New best saved: /kaggle/working/vittiny_best.pt
[polish] Ep03 | loss=1.1469 | val micro-AUC=0.7424 | micro-F1=0.1553 | 21.1m
Polish pass done.


# Step D5 (refresh) — tune thresholds on val, then evaluate test with polished checkpoint


In [21]:
import numpy as np, torch, json
from sklearn.metrics import roc_auc_score, f1_score

CKPT_PATH = "/kaggle/working/vittiny_best.pt"
device = torch.device("cpu")

# reload best weights after polish
sd = torch.load(CKPT_PATH, map_location="cpu")["state_dict"]
model.load_state_dict(sd, strict=False)
model.eval()

def predict(loader, tta=False):
    Ps, Ys = [], []
    with torch.no_grad():
        for xb, yb in loader:
            if tta:
                logits1 = model(xb.to(device)).cpu().numpy()
                logits2 = model(torch.flip(xb, dims=[3]).to(device)).cpu().numpy()
                probs = (1/(1+np.exp(-logits1)) + 1/(1+np.exp(-logits2)))/2
            else:
                logits = model(xb.to(device)).cpu().numpy()
                probs = 1/(1+np.exp(-logits))
            Ps.append(probs); Ys.append(yb.numpy())
    return np.vstack(Ps), np.vstack(Ys)

def metrics(P, Y, thr):
    micro_auc = roc_auc_score(Y, P, average='micro')
    aucs=[roc_auc_score(Y[:,j], P[:,j]) for j in range(P.shape[1]) if Y[:,j].sum()>0]
    macro_auc = float(np.mean(aucs))
    preds = (P>=thr).astype(int) if isinstance(thr,float) else (P>=np.asarray(thr)[None,:]).astype(int)
    micro_f1 = f1_score(Y, preds, average='micro', zero_division=0)
    macro_f1 = f1_score(Y, preds, average='macro', zero_division=0)
    return dict(microAUC=micro_auc, macroAUC=macro_auc,
                microF1=micro_f1, macroF1=macro_f1)

# --- tune thresholds on VAL ---
P_val, Y_val = predict(val_loader, tta=False)
grid = np.linspace(0.05, 0.95, 19)
thr_vec = np.zeros(P_val.shape[1], dtype=np.float32)
for j in range(P_val.shape[1]):
    if Y_val[:,j].sum()==0: thr_vec[j]=0.5; continue
    f1s = [f1_score(Y_val[:,j], (P_val[:,j]>=t).astype(int), zero_division=0) for t in grid]
    thr_vec[j] = float(grid[int(np.argmax(f1s))])

val_tuned = metrics(P_val, Y_val, thr=thr_vec)
print(f"[VAL-polished] microAUC={val_tuned['microAUC']:.4f} macroAUC={val_tuned['macroAUC']:.4f} "
      f"microF1={val_tuned['microF1']:.4f} macroF1={val_tuned['macroF1']:.4f}")

# --- TEST eval ---
P_test, Y_test = predict(test_loader, tta=False)
test_tuned = metrics(P_test, Y_test, thr=thr_vec)
print(f"[TEST-polished] microAUC={test_tuned['microAUC']:.4f} macroAUC={test_tuned['macroAUC']:.4f} "
      f"microF1={test_tuned['microF1']:.4f} macroF1={test_tuned['macroF1']:.4f}")

# --- TEST+TTA eval ---
P_test_tta, _ = predict(test_loader, tta=True)
test_tuned_tta = metrics(P_test_tta, Y_test, thr=thr_vec)
print(f"[TEST+TTA-polished] microAUC={test_tuned_tta['microAUC']:.4f} macroAUC={test_tuned_tta['macroAUC']:.4f} "
      f"microF1={test_tuned_tta['microF1']:.4f} macroF1={test_tuned_tta['macroF1']:.4f}")

# save updated metrics + thresholds
with open("/kaggle/working/metrics_final_polished.json","w") as f:
    json.dump({"val_tuned": val_tuned,
               "test_tuned": test_tuned,
               "test_tuned_tta": test_tuned_tta,
               "thresholds": {c: float(t) for c,t in zip(CLASSES, thr_vec)}}, f, indent=2)
print("Saved: metrics_final_polished.json")


[VAL-polished] microAUC=0.7799 macroAUC=0.7513 microF1=0.2639 macroF1=0.1976
[TEST-polished] microAUC=0.7715 macroAUC=0.7473 microF1=0.2507 macroF1=0.1851
[TEST+TTA-polished] microAUC=0.7740 macroAUC=0.7477 microF1=0.2523 macroF1=0.1847
Saved: metrics_final_polished.json


In [33]:
import torch, shutil, json, zipfile, os

# 1) Save polished model checkpoint
FINAL_MODEL_PATH = "/kaggle/working/vittiny_final.pt"
best_sd = torch.load("/kaggle/working/vittiny_best.pt", map_location="cpu")["state_dict"]
torch.save({"state_dict": best_sd, "classes": CLASSES}, FINAL_MODEL_PATH)
print("✅ Final model saved to:", FINAL_MODEL_PATH)

# 2) Collect all key files
files_to_keep = [
    "vittiny_final.pt",
    "train.csv", "val.csv", "test.csv",
    "pos_weight.json",
    "metrics_final_polished.json",
    "thresholds.json"
]

# ensure files exist in /kaggle/working
for f in files_to_keep:
    src = f"/kaggle/working/{f}"
    if not os.path.exists(src):
        raise FileNotFoundError(f"{src} not found!")

# 3) Build manifest
manifest = {
    "model_file": "vittiny_final.pt",
    "splits": ["train.csv","val.csv","test.csv"],
    "pos_weight": "pos_weight.json",
    "metrics": "metrics_final_polished.json",
    "thresholds": "thresholds.json",
    "classes": CLASSES
}
with open("/kaggle/working/project_manifest.json","w") as f:
    json.dump(manifest, f, indent=2)

# 4) Zip everything
zip_path = "/kaggle/working/vittiny_package.zip"
with zipfile.ZipFile(zip_path, "w") as z:
    for fname in files_to_keep + ["project_manifest.json"]:
        z.write(f"/kaggle/working/{fname}", arcname=fname)
print("✅ Created archive:", zip_path)


✅ Final model saved to: /kaggle/working/vittiny_final.pt
✅ Created archive: /kaggle/working/vittiny_package.zip


In [35]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def eval_classic(P, Y, thr=0.5):
    # threshold
    preds = (P >= thr).astype(int)
    
    # Micro
    acc_micro = accuracy_score(Y.flatten(), preds.flatten())
    prec_micro = precision_score(Y.flatten(), preds.flatten(), average='micro', zero_division=0)
    rec_micro = recall_score(Y.flatten(), preds.flatten(), average='micro', zero_division=0)
    f1_micro = f1_score(Y.flatten(), preds.flatten(), average='micro', zero_division=0)
    
    # Macro
    prec_macro = precision_score(Y, preds, average='macro', zero_division=0)
    rec_macro = recall_score(Y, preds, average='macro', zero_division=0)
    f1_macro = f1_score(Y, preds, average='macro', zero_division=0)
    
    return {
        "Accuracy": acc_micro,
        "Presicion - Micro": prec_micro,
        "Recall - Micro": rec_micro,
        "F1-Score - Micro": f1_micro,
        "Presicion -  Macro": prec_macro,
        "Recall - Macro": rec_macro,
        "F1-Score - Macro": f1_macro,
    }

# --- Run on TEST with tuned thresholds ---
P_test, Y_test = predict(test_loader, tta=False)   # reuse predict() from earlier
classic_metrics = eval_classic(P_test, Y_test, thr=thr_vec)

print("\n=== Classic Metrics on Test (with tuned thresholds) ===")
for k,v in classic_metrics.items():
    print(f"{k}: {v:.4f}")



=== Classic Metrics on Test (with tuned thresholds) ===
Accuracy: 0.8722
Presicion - Micro: 0.8722
Recall - Micro: 0.8722
F1-Score - Micro: 0.8722
Presicion -  Macro: 0.1355
Recall - Macro: 0.3267
F1-Score - Macro: 0.1851


# ViT done!